In [ ]:
!pip uninstall -y flash-attn
!pip install "transformers<4.47.0" "peft<0.14.0" "ms-swift[llm]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!unzip images.zip

Archive:  images.zip
   creating: images/
  inflating: images/image478.jpg     
  inflating: __MACOSX/images/._image478.jpg  
  inflating: images/image336.jpg     
  inflating: __MACOSX/images/._image336.jpg  
  inflating: images/image450.jpg     
  inflating: __MACOSX/images/._image450.jpg  
  inflating: images/image444.jpg     
  inflating: __MACOSX/images/._image444.jpg  
  inflating: images/image322.jpg     
  inflating: __MACOSX/images/._image322.jpg  
  inflating: images/image493.jpg     
  inflating: __MACOSX/images/._image493.jpg  
  inflating: images/image487.jpg     
  inflating: __MACOSX/images/._image487.jpg  
  inflating: images/image108.jpg     
  inflating: __MACOSX/images/._image108.jpg  
  inflating: images/image652.jpg     
  inflating: __MACOSX/images/._image652.jpg  
  inflating: images/image134.jpg     
  inflating: __MACOSX/images/._image134.jpg  
  inflating: images/image120.jpg     
  inflating: __MACOSX/images/._image120.jpg  
  inflating: images/image646.jpg  

In [ ]:
!jar xvf V2Images2.zip

  created: V2Images/
java.util.zip.ZipException: ZipFile invalid LOC header (bad signature)
	at java.base/java.util.zip.ZipFile$ZipFileInputStream.initDataOffset(ZipFile.java:928)
	at java.base/java.util.zip.ZipFile$ZipFileInputStream.read(ZipFile.java:939)
	at java.base/java.util.zip.ZipFile$ZipFileInflaterInputStream.fill(ZipFile.java:456)
	at java.base/java.util.zip.InflaterInputStream.read(InflaterInputStream.java:158)
	at java.base/java.io.FilterInputStream.read(FilterInputStream.java:106)
	at jdk.jartool/sun.tools.jar.Main.copy(Main.java:1254)
	at jdk.jartool/sun.tools.jar.Main.copy(Main.java:1282)
	at jdk.jartool/sun.tools.jar.Main.extractFile(Main.java:1457)
	at jdk.jartool/sun.tools.jar.Main.extract(Main.java:1392)
	at jdk.jartool/sun.tools.jar.Main.run(Main.java:389)
	at jdk.jartool/sun.tools.jar.Main.main(Main.java:1681)


In [ ]:
import json
def build_jsonl(input_file,output_file):
  data = json.load(open(input_file))
  with open(output_file,'w',encoding = 'utf-8') as f:
    for item in data:
      for entry in item["conversations"]:
        if entry["from"] == "assistant" and isinstance(entry["value"], dict):
          entry["value"] = json.dumps(entry["value"], ensure_ascii=False)
      json_record = json.dumps(item,ensure_ascii=False)
      f.write(json_record + '\n')
  print(f"✅ Done! {len(data)} samples converted to {output_file}")

In [ ]:
build_jsonl('Fine_Tune1.json','FineTune.jsonl')

✅ Done! 1003 samples converted to FineTune.jsonl


In [ ]:
import json
from sklearn.model_selection import train_test_split
from collections import Counter

with open('FineTune.jsonl', 'r') as f:
    lines = f.readlines()
data = [json.loads(line) for line in lines]

stratify_labels = []
for item in data:
    try:
        content = item['conversations'][1]['value']
        if isinstance(content, str):
            val = json.loads(content)
        else:
            val = content
        t = val['labels'].get('type', 'none')
        stratify_labels.append(t)
    except:
        stratify_labels.append('none')

counts = Counter(stratify_labels)
main_indices = [i for i, l in enumerate(stratify_labels) if counts[l] > 1]
loner_indices = [i for i, l in enumerate(stratify_labels) if counts[l] <= 1]

train_idx, test_idx = train_test_split(
    main_indices, test_size=0.20, stratify=[stratify_labels[i] for i in main_indices], random_state=42
)

train_data = [data[i] for i in train_idx] + [data[i] for i in loner_indices]
test_data = [data[i] for i in test_idx]

with open('train_split.jsonl', 'w') as f:
    for item in train_data: f.write(json.dumps(item) + '\n')
with open('test_split.jsonl', 'w') as f:
    for item in test_data: f.write(json.dumps(item) + '\n')

print(f"✅ Stratified Split Done! Train: {len(train_data)} | Test: {len(test_data)}")

✅ Stratified Split Done! Train: 802 | Test: 201


In [ ]:
import json

dataset_info = [
    {
        "ms_dataset_id": "meme-train",
        "dataset_path": "train_split.jsonl",
        "columns": {
            "conversations": "messages", 
            "images": "images",
            "system": "system"
        },
        "preprocess_func": "sharegpt"
    },
    {
        "ms_dataset_id": "meme-val", 
        "dataset_path": "test_split.jsonl",
        "columns": {
            "conversations": "messages",
            "images": "images",
            "system": "system"
        },
        "preprocess_func": "sharegpt"
    }
]

with open("dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

print("✅ dataset_info.json is now synced with your command!")

✅ dataset_info.json is now synced with your command!


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download(repo_id="OpenGVLab/InternVL2_5-8B", local_dir="internvl2_5_8b", local_dir_use_symlinks=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

conversation.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

...

runs/Nov18_19-03-50_HOST-10-140-60-23/ev(…):   0%|          | 0.00/854k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenization_internlm2.py: 0.00B [00:00, ?B/s]

tokenization_internlm2_fast.py: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

In [ ]:
!swift sft \
    --model /content/internvl2_5_8b \
    --model_type internvl2_5 \
    --train_type lora \
    --dataset train_split.jsonl \
    --val_dataset test_split.jsonl \
    --torch_dtype bfloat16 \
    --attn_impl eager \
    --device_map auto \
    --lazy_tokenize true \
    --output_dir output_meme_expert_r32 \
    --num_train_epochs 3 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-5 \
    --lora_rank 32 \
    --lora_alpha 64 \
    --target_modules all-linear \
    --gradient_checkpointing true \
    --eval_steps 50 \
    --save_steps 50 \
    --save_total_limit 2 \
    --load_best_model_at_end true \
    --metric_for_best_model loss

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-03-20 08:35:58.228611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 08:35:58.247366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773995758.270389    4824 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773995758.278007    4824 cuda_blas.cc:1407] Unable to reg

In [ ]:
!swift sft \
    --model /content/internvl2_5_8b \
    --model_type internvl2_5 \
    --dataset train_split.jsonl \
    --val_dataset test_split.jsonl \
    --tuner_type lora \
    --torch_dtype bfloat16 \
    --attn_impl eager \
    --device_map auto \
    --output_dir output_meme_expert_final \
    --num_train_epochs 5 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --learning_rate 2e-5 \
    --lora_rank 32 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --freeze_vit false \
    --freeze_aligner false \
    --acc_strategy seq \
    --gradient_checkpointing true \
    --eval_steps 50 \
    --save_steps 50 \
    --save_total_limit 2 \
    --load_best_model_at_end true \
    --metric_for_best_model loss

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-03-21 15:23:54.921934: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 15:23:54.939927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774106634.962226    5315 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774106634.969542    5315 cuda_blas.cc:1407] Unable to reg

In [ ]:
import json
import os

# Do this for both checkpoint folders
checkpoints = [
    "/content/output_meme_expert_final/v0-20260321-152413/checkpoint-200",
    "/content/output_meme_expert_final/v0-20260321-152413/checkpoint-250"
]

for ckpt in checkpoints:
    config_path = os.path.join(ckpt, "adapter_config.json")

    with open(config_path, 'r') as f:
        config = json.load(f)

    # Change the local path to the Hugging Face model ID
    config["base_model_name_or_path"] = "OpenGVLab/InternVL2_5-8B"

    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

print("✅ Fixed adapter_config.json in both folders!")

✅ Fixed adapter_config.json in both folders!


In [ ]:
for ckpt in checkpoints:
    readme_path = os.path.join(ckpt, "README.md")
    if os.path.exists(readme_path):
        os.remove(readme_path)
        print(f"🗑️ Removed local README from {ckpt}")

🗑️ Removed local README from /content/output_meme_expert_final/v0-20260321-152413/checkpoint-200
🗑️ Removed local README from /content/output_meme_expert_final/v0-20260321-152413/checkpoint-250


In [ ]:
from huggingface_hub import HfApi, create_repo

# 1. PASTE YOUR TOKEN HERE
MY_TOKEN = ""

api = HfApi(token=MY_TOKEN)
repo_id = "debarghaNath/BullyingMemeDetector"

print(f"🛠️ Ensuring repo exists: {repo_id}")
create_repo(repo_id=repo_id, token=MY_TOKEN, exist_ok=True, repo_type="model")


print("⬆️ Uploading BEST model (checkpoint-200)...")
api.upload_folder(
    folder_path="/content/output_meme_expert_final/v0-20260321-152413/checkpoint-200",
    repo_id=repo_id,
    path_in_repo="1000_final_best_model",
    repo_type="model",
    token=MY_TOKEN
)


print("⬆️ Uploading LAST model (checkpoint-30-)...")
api.upload_folder(
    folder_path="/content/output_meme_expert_final/v0-20260321-152413/checkpoint-250",
    repo_id=repo_id,
    path_in_repo="1000_final_last_model",
    repo_type="model",
    token=MY_TOKEN
)

print(f"🎉 Success! Check your models at: https://huggingface.co/{repo_id}")

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
with open('test_split.jsonl', 'r') as f:
    test_items = [json.loads(line) for line in f]

In [ ]:
import json
import pandas as pd
import re 
from tqdm import tqdm

results_list = []

print(f"🚀 Starting final evaluation on {len(test_items)} images...")

for item in tqdm(test_items):
    test_img = item['images'][0]

    try:
        raw_output = analyze_meme(test_img)
        match = re.search(r'\{.*\}', raw_output, re.DOTALL)

        if not match:
            raise ValueError("No JSON found in model output")

        json_str = match.group(0)

        data = json.loads(json_str)

        if 'labels' in data:
            labels = data.get('labels', {})
            analysis = data.get('analysis', {})
        else:
            labels = data
            analysis = data

        results_list.append({
            "image_path": test_img,
            "pred_bullying": labels.get('bullying'),
            "pred_target": labels.get('target'),
            "pred_mechanism": labels.get('mechanism'),
            "pred_type": labels.get('type'),
            "pred_severity": labels.get('severity'),
            "visual_cues": analysis.get('visual_cues', 'N/A'),
            "text_cues": analysis.get('text_cues', 'N/A'),
            "interpretation": analysis.get('interpretation', 'N/A')
        })

    except Exception as e:
        results_list.append({
            "image_path": test_img,
            "pred_bullying": "ERROR",
            "interpretation": f"Failed to parse: {str(e)}",
            "raw_text_debug": raw_output[:100] 
        })

df = pd.DataFrame(results_list)
df.to_csv("bullying_analysis_results.csv", index=False)
print("\n✅ Loop Complete! Data saved to bullying_analysis_results.csv")

🚀 Starting final evaluation on 201 images...


100%|██████████| 201/201 [09:45<00:00,  2.91s/it]


✅ Loop Complete! Data saved to bullying_analysis_results.csv


In [ ]:
result = analyze_meme("/content/image1004.jpg")
print(result)

```json
{
  "bullying": "No",
  "target": "individual",
  "mechanism": "sarcastic_dissonance",
  "type": "physical_appearance",
  "severity": "low"
}
```


In [ ]:
# --- TEST IT ---
image_to_test = "image124.jpg"
print("\n--- ANALYZING MEME ---")
result = analyze_meme(image_to_test)
print(result)


--- ANALYZING MEME ---
 {"analysis": {"visual_cues": "The image features a woman posing with her back to the camera, her right hand on her hip, and her head turned to the side, revealing her profile.", "text_cues": "The text overlay reads: 'THE 'I HAVE NOTHING ELSE TO OFFER' POSE' in all capital letters.", "text_location": "embedded_in_image", "interpretation": "The content utilizes coded symbolism to mock and shame women for engaging in self-promotion or self-admiration. By labeling a common pose as 'having nothing else to offer,' the creator employs a form of normative comparison to devalue the subject's agency and self-worth. This is a form of relational social bullying that targets women's bodies and their perceived value in the context of societal standards of beauty and sexuality.", "confidence": "high"}, "labels": {"bullying": "Yes", "target": "demographic_group", "mechanism": "normative_comparison", "type": "relational_social", "severity": "high"}}<|im_end|>


In [ ]:
# --- TEST IT ---
image_to_test = "image71.jpg"
print("\n--- ANALYZING MEME ---")
result = analyze_meme(image_to_test)
print(result)


--- ANALYZING MEME ---
 {"analysis": {"visual_cues": "The image features a woman with a disapproving or disappointed facial expression, resting her chin on her hand.", "text_cues": "The text reads: 'Every MOM's Reaction' and 'When they See their Son/Daughter Speak in English to Someone.'", "text_location": "embedded_in_image", "interpretation": "The content utilizes sarcastic dissonance to mock the perceived hypocrisy of certain groups. By juxtaposing the 'disapproving mother' trope with the scenario of a child speaking English, the meme implies that the reaction is exaggerated or unwarranted. This can be seen as a form of normative comparison, where the 'normal' reaction is being ridiculed for being 'overly sensitive' or 'culturally superior.'", "confidence": "high"}, "labels": {"bullying": "Yes", "target": "demographic_group", "mechanism": "sarcastic_dissonance", "type": "religious_political", "severity": "medium"}}<|im_end|>


In [ ]:
# --- TEST IT ---
image_to_test = "image94.jpg"
print("\n--- ANALYZING MEME ---")
result = analyze_meme(image_to_test)
print(result)


--- ANALYZING MEME ---
 {"analysis": {"visual_cues": "The image is a still from the animated show 'SpongeBob SquarePants'. It features the character Squidward standing on a boat, looking disheveled and exhausted, with red marks on his face and a bloodied nose. The character Patrick Star is standing next to him, smiling broadly and holding a picnic basket.", "text_cues": "The text overlay reads: 'Me liking my depressed friend's 3am whatsapp status'."}, "text_location": "embedded_in_image", "interpretation": "The content utilizes sarcastic dissonance to mock the act of 'liking' a friend's late-night social media posts. By juxtaposing the innocent and cheerful character Patrick with the visibly distressed Squidward, the meme employs normative comparison to highlight the absurdity of 'liking' a friend's 'depressed' status. This creates a humorous critique of social media behavior, particularly the tendency to engage with content that is ostensibly'relatable' or 'trendy' without genuine em

In [ ]:
# --- TEST IT ---
image_to_test = "image97.jpg"
print("\n--- ANALYZING MEME ---")
result = analyze_meme(image_to_test)
print(result)


--- ANALYZING MEME ---
 {"analysis": {"visual_cues": "The image is a meme featuring a popular 'North Korean' style of humor. It shows a caricature of Kim Jong-un, the leader of North Korea, whispering to another caricature of a North Korean official. The official is wearing a military uniform with a red beret and a gold epaulet. The background is a blurred image of a North Korean propaganda poster.", "text_cues": "The text reads 'LAUNCH? I SAID LUNCH.'", "text_location": "embedded_in_image", "interpretation": "The meme uses coded symbolism to mock the North Korean regime's potential for launching nuclear weapons. By replacing 'launch' with 'lunch,' the meme employs a form of sarcastic dissonance to highlight the absurdity of the regime's actions. The use of a military uniform and a propaganda poster in the background further emphasizes the regime's authoritarian and militaristic nature.", "confidence": "high"}, "labels": {"bullying": "Yes", "target": "organization", "mechanism": "sarc

In [ ]:
import swift
print(swift.__version__)


4.0.2
